# Homework 3

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import train_test_split

In [56]:
%%script true # Skips the cell to prevent repeated dataset downloads.

data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

!wget $data

In [57]:
df_raw = pd.read_csv('course_lead_scoring_2026.csv')

In [58]:
df = df_raw.copy()
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


## Data Preparation

In [59]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [60]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [61]:
categorical_features = list(df.dtypes[df.dtypes == 'str'].index)
categorical_features

['lead_source', 'industry', 'employment_status', 'location']

In [62]:
numerical_features = list(df.dtypes[(df.dtypes != 'str') & (df.dtypes.index != 'converted')].index)
numerical_features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [63]:
df[categorical_features] = df[categorical_features].fillna('NA')

df[numerical_features] = df[numerical_features].fillna(0.0)

In [64]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1

##### What is the most frequent observation (mode) for the column industry?

    NA
    technology
    healthcare
    retail

In [65]:
industry_mode = df.industry.mode().iloc[0]

print(f'The most frequent observation mode for the column industry is {industry_mode}.')

The most frequent observation mode for the column industry is technology.


## Question 2

##### Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

###### What are the two features that have the biggest correlation?

    interaction_count and lead_score
    number_of_courses_viewed and lead_score
    number_of_courses_viewed and interaction_count
    annual_income and interaction_count
    
##### Only consider the pairs above when answering this question.

In [73]:
correlation_matrix = df[numerical_features].corr()

correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [74]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs:
    score = correlation_matrix.loc[col1, col2]
    print(f'{col1} and {col2}: {score:.4f}')

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [75]:
strongest_pair = max(
    pairs,
    key=lambda pair: abs(correlation_matrix.loc[pair[0], pair[1]])
)

print('Strongest pair:', strongest_pair)

Strongest pair: ('interaction_count', 'lead_score')


### Data Split

In [79]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

In [80]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [81]:
del df_train['converted']
del df_val['converted']

## Question 3

    Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
    Round the scores to 2 decimals using round(score, 2).
##### Which of these variables has the biggest mutual information score?

    industry
    location
    lead_source
    employment_status